## ML_1042: Top-k / Top-p Sampling

**Difficulty**: Medium | **TorchCode**: #32

### Problem
Implement token sampling with temperature, top-k filtering, and top-p (nucleus) filtering. Apply temperature first, then top-k (keep k largest logits), then top-p (keep smallest set covering p probability mass), then sample from softmax.

### Formula
$$p_i = \text{softmax}(l_i / T), \quad \text{top-k: } l_i = -\infty \text{ if } l_i < l_{(k)}, \quad \text{top-p: mask where cumsum} > p$$

In [ ]:
import torch

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    logits = logits / max(temperature, 1e-8)
    if top_k > 0:
        top_k_val = logits.topk(top_k).values[-1]
        logits[logits < top_k_val] = float('-inf')
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = torch.softmax(sorted_logits, dim=-1)
        cumsum = torch.cumsum(probs, dim=-1)
        mask = (cumsum - probs) > top_p
        sorted_logits[mask] = float('-inf')
        logits = torch.empty_like(logits).scatter_(0, sorted_idx, sorted_logits)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()

In [ ]:
import torch

# --- Test 1: top_k=1 always returns argmax ---
torch.manual_seed(0)
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
for _ in range(10):
    assert sample_top_k_top_p(logits.clone(), top_k=1) == 1

# --- Test 2: Low temperature concentrates on argmax ---
torch.manual_seed(42)
logits = torch.tensor([1.0, 3.0, 2.0])
counts = [0, 0, 0]
for _ in range(100):
    counts[sample_top_k_top_p(logits.clone(), temperature=0.01)] += 1
assert counts[1] > 90

# --- Test 3: All tokens reachable with no filtering ---
logits = torch.zeros(5)
seen = set()
for i in range(200):
    torch.manual_seed(i)
    seen.add(sample_top_k_top_p(logits.clone()))
assert len(seen) == 5

# --- Test 4: Returns valid index ---
torch.manual_seed(0)
V = 100
logits = torch.randn(V)
for _ in range(20):
    t = sample_top_k_top_p(logits.clone(), top_k=10, top_p=0.9)
    assert 0 <= t < V

print('All tests passed!')